In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [20]:
PTSmodel = joblib.load('Models/xgbPTSModel.pkl')
PTSfeatures = joblib.load('Models/topPTSfeatures.pkl')

### Load Data

In [ ]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = single_bet(
    data=s25,
    bookmakers=singlePTSBookies,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=10000, 
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
singleBets = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets_{today}.csv', index=False)

Processing single bets...


## Top EVs for 2 leg bets

### Underdog picks

In [15]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = prizepickspairsEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
underdogPairs = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs_{today}.csv', index=False)
underdogPairs.head()

Processing pairs...


,PLAYER 1,PLAYER 2,CATEGORY,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL_SIDE 1,MODEL_SIDE 2,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Aaron Gordon,Gary Payton II,player_points,14.5,4.5,17.22,10.62,OVER,OVER,0.626,0.374,0.950,0.050,"(0.9, 32.2)","(4.6, 16.3)",OVER/OVER,0,0.5947,0.784,0.392
1,Aaron Gordon,Max Christie,player_points,14.5,5.5,17.22,13.22,OVER,OVER,0.626,0.374,0.950,0.050,"(0.9, 32.2)","(6.0, 20.4)",OVER/OVER,0,0.5947,0.784,0.392
2,Aaron Gordon,Collin Gillespie,player_points,14.5,7.5,17.22,13.09,OVER,OVER,0.626,0.374,0.946,0.054,"(0.9, 32.2)","(6.6, 19.6)",OVER/OVER,0,0.5922,0.777,0.388
3,Aaron Gordon,Julian Champagnie,player_points,14.5,9.5,17.22,17.24,OVER,OVER,0.626,0.374,0.935,0.065,"(0.9, 32.2)","(7.9, 26.2)",OVER/OVER,0,0.5853,0.756,0.378
4,Aaron Gordon,Harrison Barnes,player_points,14.5,9.5,17.22,17.21,OVER,OVER,0.626,0.374,0.932,0.068,"(0.9, 32.2)","(6.1, 27.9)",OVER/OVER,0,0.5834,0.750,0.375


### Prizepicks picks

In [17]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = prizepickspairsEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
pairsPrizepicks = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs_{today}.csv', index=False)
pairsPrizepicks.head()

Processing pairs...


,PLAYER 1,PLAYER 2,CATEGORY,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL_SIDE 1,MODEL_SIDE 2,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Aaron Gordon,Drew Eubanks,player_points,14.5,8.5,17.22,3.10,OVER,UNDER,0.611,0.389,0.050,0.950,"(2.6, 32.5)","(0.0, 8.2)",OVER/UNDER,0,0.5804,0.741,0.371
1,Aaron Gordon,Max Christie,player_points,14.5,5.5,17.22,13.22,OVER,OVER,0.611,0.389,0.950,0.050,"(2.6, 32.5)","(5.5, 20.7)",OVER/OVER,0,0.5804,0.741,0.371
2,Aaron Gordon,Julian Champagnie,player_points,14.5,9.5,17.22,17.24,OVER,OVER,0.611,0.389,0.939,0.061,"(2.6, 32.5)","(7.4, 26.8)",OVER/OVER,0,0.5737,0.721,0.361
3,Aaron Gordon,Stephon Castle,player_points,14.5,15.5,17.22,25.18,OVER,OVER,0.611,0.389,0.925,0.075,"(2.6, 32.5)","(12.8, 38.2)",OVER/OVER,0,0.5652,0.696,0.348
4,Aaron Gordon,Harrison Barnes,player_points,14.5,9.0,17.22,17.21,OVER,OVER,0.611,0.389,0.912,0.088,"(2.6, 32.5)","(5.8, 27.6)",OVER/OVER,0,0.5572,0.672,0.336


### DraftKings Pick 6 picks

In [23]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'DraftKings Pick6') & (dfsData['CATEGORY'] == 'player_points')]

results = prizepickspairsEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
pairsDraftKings = results.sort_values(by='EV%', ascending=False).reset_index(drop=True)
pairsDraftKings.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/draftKingsPairs_{today}.csv', index=False)
pairsDraftKings.head()

Processing pairs...


,PLAYER 1,PLAYER 2,CATEGORY,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL_SIDE 1,MODEL_SIDE 2,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Anthony Davis,Caris LeVert,player_points,24.5,8.5,25.1,17.44,OVER,OVER,0.541,0.459,0.95,0.05,"(10.9, 40.1)","(7.6, 27.1)",OVER/OVER,0,0.514,0.542,0.271
1,Anthony Davis,Drew Eubanks,player_points,24.5,8.5,25.1,3.10,OVER,UNDER,0.541,0.459,0.05,0.95,"(10.9, 40.1)","(0.0, 8.4)",OVER/UNDER,0,0.514,0.542,0.271
2,Anthony Davis,Joel Embiid,player_points,24.5,6.5,25.1,27.81,OVER,OVER,0.541,0.459,0.95,0.05,"(10.9, 40.1)","(12.9, 43.6)",OVER/OVER,0,0.514,0.542,0.271
3,Anthony Davis,Kyle Filipowski,player_points,24.5,6.5,25.1,17.70,OVER,OVER,0.541,0.459,0.95,0.05,"(10.9, 40.1)","(5.9, 29.5)",OVER/OVER,0,0.514,0.542,0.271
4,Anthony Davis,Walker Kessler,player_points,24.5,20.5,25.1,10.90,OVER,UNDER,0.541,0.459,0.05,0.95,"(10.9, 40.1)","(0.0, 21.2)",OVER/UNDER,0,0.514,0.542,0.271


## 3 leg parlay

### Underdog picks

In [18]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = prizepicks3LegEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=10000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
underdogTrios = threeLeg.sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios_{today}.csv', index=False)
underdogTrios.head()

Processing 3-leg parlays...


,PLAYER 1,PLAYER 2,PLAYER 3,CATEGORY,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_SIDE 1,MODEL_SIDE 2,MODEL_SIDE 3,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Collin Gillespie,Gary Payton II,Max Christie,player_points,7.5,4.5,5.5,13.09,10.62,13.22,OVER,OVER,OVER,0.946,0.054,0.950,0.050,0.950,0.050,"(6.3, 19.6)","(5.0, 16.3)","(6.1, 20.3)",OVER/OVER/OVER,1,0.8533,4.120,0.824
1,Gary Payton II,Julian Champagnie,Max Christie,player_points,4.5,9.5,5.5,10.62,17.24,13.22,OVER,OVER,OVER,0.950,0.050,0.941,0.059,0.950,0.050,"(5.0, 16.3)","(7.4, 27.1)","(6.1, 20.3)",OVER/OVER/OVER,1,0.8488,4.093,0.819
2,Collin Gillespie,Julian Champagnie,Max Christie,player_points,7.5,9.5,5.5,13.09,17.24,13.22,OVER,OVER,OVER,0.946,0.054,0.941,0.059,0.950,0.050,"(6.3, 19.6)","(7.4, 27.1)","(6.1, 20.3)",OVER/OVER/OVER,1,0.8448,4.069,0.814
3,Collin Gillespie,Gary Payton II,Julian Champagnie,player_points,7.5,4.5,9.5,13.09,10.62,17.24,OVER,OVER,OVER,0.946,0.054,0.950,0.050,0.941,0.059,"(6.3, 19.6)","(5.0, 16.3)","(7.4, 27.1)",OVER/OVER/OVER,1,0.8448,4.069,0.814
4,Gary Payton II,Harrison Barnes,Max Christie,player_points,4.5,9.5,5.5,10.62,17.21,13.22,OVER,OVER,OVER,0.950,0.050,0.922,0.078,0.950,0.050,"(5.0, 16.3)","(6.3, 27.9)","(6.1, 20.3)",OVER/OVER/OVER,1,0.8318,3.991,0.798


### Prizepicks picks

In [19]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = prizepicks3LegEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=10000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
triosPrizepicks = threeLeg.sort_values(by='EV%', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios_{today}.csv', index=False)
triosPrizepicks.head()

Processing 3-leg parlays...


,PLAYER 1,PLAYER 2,PLAYER 3,CATEGORY,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_SIDE 1,MODEL_SIDE 2,MODEL_SIDE 3,OVER% 1,UNDER% 1,OVER% 2,UNDER% 2,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Drew Eubanks,Julian Champagnie,Max Christie,player_points,8.5,9.5,5.5,3.10,17.24,13.22,UNDER,OVER,OVER,0.050,0.950,0.937,0.063,0.950,0.050,"(0.0, 8.1)","(7.4, 27.2)","(6.1, 20.2)",UNDER/OVER/OVER,0,0.8454,4.072,0.814
1,Drew Eubanks,Max Christie,Stephon Castle,player_points,8.5,5.5,15.5,3.10,13.22,25.18,UNDER,OVER,OVER,0.050,0.950,0.950,0.050,0.935,0.065,"(0.0, 8.1)","(6.1, 20.2)","(12.8, 37.6)",UNDER/OVER/OVER,0,0.8437,4.062,0.812
2,Drew Eubanks,Harrison Barnes,Max Christie,player_points,8.5,9.0,5.5,3.10,17.21,13.22,UNDER,OVER,OVER,0.050,0.950,0.928,0.072,0.950,0.050,"(0.0, 8.1)","(6.3, 27.9)","(6.1, 20.2)",UNDER/OVER/OVER,0,0.8377,4.026,0.805
3,Julian Champagnie,Max Christie,Stephon Castle,player_points,9.5,5.5,15.5,17.24,13.22,25.18,OVER,OVER,OVER,0.937,0.063,0.950,0.050,0.935,0.065,"(7.4, 27.2)","(6.1, 20.2)","(12.8, 37.6)",OVER/OVER/OVER,1,0.8319,3.992,0.798
4,Drew Eubanks,Julian Champagnie,Stephon Castle,player_points,8.5,9.5,15.5,3.10,17.24,25.18,UNDER,OVER,OVER,0.050,0.950,0.937,0.063,0.935,0.065,"(0.0, 8.1)","(7.4, 27.2)","(12.8, 37.6)",UNDER/OVER/OVER,0,0.8319,3.992,0.798


### DraftKings Pick 6 picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'DraftKings Pick6') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = prizepicks3LegEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5.5,
    stake=100,
    simulations=10000,
    std_window=15,
    min_std=1.5,
    max_std=8.0,
    stat_col='PTS'
)
triosDraftKings = threeLeg.sort_values(by='EV%', ascending=False).reset_index(drop=True)
triosDraftKings.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/draftKingsTrios_{today}.csv', index=False)
triosDraftKings.head()